In [12]:
from openai import OpenAI
client = OpenAI()

In [13]:
class OpenAIConnector:
    def __init__(self, token_path="../credentials/open_ai_token.txt"):
        self.token = self._load_token(token_path)
        self.client = OpenAI(api_key=self.token)

    @staticmethod
    def _load_token(token_path):
        try:
            with open(token_path, "r") as f:
                token = f.read().strip()
                if not token:
                    raise ValueError("Token file is empty.")
                return token
        except FileNotFoundError:
            raise FileNotFoundError(f"Token file not found at {token_path}")
        except Exception as e:
            raise RuntimeError(f"Error reading token: {e}")

    def make_request(self, model="o4-mini", input_text="Convert 1 kg to lbs"):
        response = self.client.responses.create(
            model=model,
            input=input_text
        )
        return response.output_text
    def get_usage_summary(self):
        headers = {
            "Authorization": f"Bearer {self.token}"
        }

        now = datetime.now(timezone.utc)
        start_date = now.replace(day=1).strftime("%Y-%m-%d")
        end_date = now.strftime("%Y-%m-%d")

        # 1. Get usage data
        usage_url = f"https://api.openai.com/v1/dashboard/billing/usage?start_date={start_date}&end_date={end_date}"
        usage_resp = requests.get(usage_url, headers=headers)
        usage_data = usage_resp.json()
        used_usd = usage_data.get("total_usage", 0) / 100.0  # from cents to dollars

        # 2. Get allowance (subscription limit)
        limits_url = "https://api.openai.com/v1/dashboard/billing/subscription"
        limits_resp = requests.get(limits_url, headers=headers)
        limits_data = limits_resp.json()
        hard_limit = limits_data.get("hard_limit_usd", 0)
        soft_limit = limits_data.get("soft_limit_usd", 0)

        remaining = hard_limit - used_usd

        return {
            "used_usd": round(used_usd, 2),
            "soft_limit_usd": round(soft_limit, 2),
            "hard_limit_usd": round(hard_limit, 2),
            "remaining_usd": round(remaining, 2)
        }

In [14]:
import pandas as pd
df = pd.read_csv("../data/kaggle_restaurant_data.csv")
df.head()

/var/folders/59/yzs1t5953m12vkbtx3y5srbw0000gn/T/ipykernel_82215/771314455.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/kaggle_restaurant_data.csv")


,id,restaurant_name,score,ratings,restaurant_type,full_address,menu_category,menu_name,menu_item_description,price
0,5.0,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches","314 17th St N, Birmingham, AL, 35203",Picked for you,Pork Chop Rice with Gravy Plate,NaN,7.00
1,5.0,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches","314 17th St N, Birmingham, AL, 35203",Picked for you,Full Sausage (2 pcs) with 2 Eggs,2 pieces.,7.25
2,5.0,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches","314 17th St N, Birmingham, AL, 35203",Picked for you,Bacon and Egg with Cheese Breakfast Sandwich,NaN,3.50
3,5.0,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches","314 17th St N, Birmingham, AL, 35203",Picked for you,Double Cheese Burger,Grilled or fried patty with cheese on a bun.,3.25
4,5.0,Nelson Brothers Cafe (17th St N),4.7,22.0,"Breakfast and Brunch, Burgers, Sandwiches","314 17th St N, Birmingham, AL, 35203",Picked for you,Full Bacon (3 pcs) with 2 Eggs,3 pieces.,7.25


## Generate menu extraction prompt

### Test Prompt
You are a data extractor. Extract from input as:  
    restaurant_name = Domino's  
    restaurant_type = pizza  
    menu_name = pepperoni pizza  
For each dish, extract these:  
    Dish cuisine (e.g. "Pizza restaurant")  
    Dish base (main ingredient, e.g. "Pizza")  
    Dish flavor (flavor, e.g. "Pepperoni")  

All outputs must follow consistent spelling and formatting: 
- Use singular form for dish base and flavor (e.g., "egg" instead of "eggs")
- Normalize abbreviations (e.g., use "BBQ" not "bbq")
- Capitalize the first letter of each dish flavor and base
- Translate all non-English dish names or terms into English before extracting features
- All spellings must follow the AP Stylebook (American English). Use Merriam-Webster Dictionary as the authority for all words.  

If the restaurant's category is missing or unclear, infer the cuisine from the dish name.  
Do not include ```json or any markdown. Output must be a raw JSON array.  

Examples:  
Input: 
```text
restaurant_name = Summer restaurant
restaurant_type = Chinese, Comfort Food
menu_name = Beef Broccoli
```
Output: 
```json
[
  {
    "dish_cuisine": "Chinese restaurant",
    "dish_base": "Beef",
    "dish_flavor": "Broccoli"
  }
]
```
Process the following inputs:  
restaurant_name =  
restaurant_type =  
menu_name =  

In [15]:
df_sample_20 = df.sample(n=20, random_state=123).reset_index(drop=True)
df_sample_20 = df_sample_20[["restaurant_name", "restaurant_type", "menu_name"]]

In [26]:
import math
import pandas as pd

# Assume BATCH_SIZE = 10
def prepare_input_batches(df: pd.DataFrame, batch_size: int) -> list[str]:
    
    # 1) System instruction
    system_inst = (
        "You are a data extractor. Extract from input as:\n"
        "    restaurant_name = Domino's\n"
        "    restaurant_type = pizza\n"
        "    menu_name = pepperoni pizza\n"
        "For each dish, extract these:\n"
        "    Dish cuisine (e.g. \"Pizza restaurant\")\n"
        "    Dish base (main ingredient, e.g. \"Pizza\")\n"
        "    Dish flavor (flavor, e.g. \"Pepperoni\")\n\n"
        "All outputs must follow consistent spelling and formatting:\n"
        "- Use singular form for dish base and flavor (e.g., \"egg\" instead of \"eggs\")\n"
        "- Normalize abbreviations (e.g., use \"BBQ\" not \"bbq\")\n"
        "- Capitalize the first letter of each dish flavor and base\n"
        "- Translate all non-English dish names or terms into English before extracting features\n"
        "If the restaurant's category is missing or unclear, infer the cuisine from the dish name.\n"
        "Do not include ```json or any markdown. Output must be a raw JSON array.\n\n"
        "Examples:\n"
        "Input:\n"
        "```text\n"
        "restaurant_name = Summer restaurant\n"
        "restaurant_type = Chinese, Comfort Food\n"
        "menu_name = Beef Broccoli\n"
        "```\n"
        "Output:\n"
        "```json\n"
        "[\n"
        "  {\n"
        "    \"dish_cuisine\": \"Chinese restaurant\",\n"
        "    \"dish_base\": \"Beef\",\n"
        "    \"dish_flavor\": \"Broccoli\"\n"
        "  }\n"
        "]\n"
        "```\n\n"
        "Process the following inputs:\n"
    )
    
    # 2) Template (user part)
    tmpl = (
        "restaurant_name = {restaurant_name}\n"
        "restaurant_type = {restaurant_type}\n"
        "menu_name = {menu_name}\n\n"
    )
    
    # 3) Generate all pieces of prompts
    pieces = [
        tmpl.format(
            menu_name=row["menu_name"],
            restaurant_name=row["restaurant_name"],
            restaurant_type=row["restaurant_type"]
        )
        for _, row in df.iterrows()
    ]
    
    # 4) Slice and generate based on batch size
    n_batches = math.ceil(len(pieces) / batch_size)
    batches = []
    for i in range(n_batches):
        start, end = i * batch_size, (i + 1) * batch_size
        user_block = "".join(pieces[start:end])
        batches.append(system_inst + user_block)
    return batches

# test 
batches = prepare_input_batches(df_sample_20, batch_size=10)
prompt_text = ''
for inp in batches:
    prompt_text += inp + '\n'
print(prompt_text)

You are a data extractor. Extract from input as:
    restaurant_name = Domino's
    restaurant_type = pizza
    menu_name = pepperoni pizza
For each dish, extract these:
    Dish cuisine (e.g. "Pizza restaurant")
    Dish base (main ingredient, e.g. "Pizza")
    Dish flavor (flavor, e.g. "Pepperoni")

All outputs must follow consistent spelling and formatting:
- Use singular form for dish base and flavor (e.g., "egg" instead of "eggs")
- Normalize abbreviations (e.g., use "BBQ" not "bbq")
- Capitalize the first letter of each dish flavor and base
- Translate all non-English dish names or terms into English before extracting features
If the restaurant's category is missing or unclear, infer the cuisine from the dish name.
Do not include ```json or any markdown. Output must be a raw JSON array.

Examples:
Input:
```text
restaurant_name = Summer restaurant
restaurant_type = Chinese, Comfort Food
menu_name = Beef Broccoli
```
Output:
```json
[
  {
    "dish_cuisine": "Chinese restaurant",


In [27]:
connector = OpenAIConnector()
response = connector.make_request(input_text=prompt_text)
print(response)

[
  {
    "dish_cuisine": "Mexican restaurant",
    "dish_base": "Quesadilla",
    "dish_flavor": "Chicken"
  },
  {
    "dish_cuisine": "American restaurant",
    "dish_base": "Slider",
    "dish_flavor": "Kid's"
  },
  {
    "dish_cuisine": "Chinese restaurant",
    "dish_base": "Lo Mein",
    "dish_flavor": "Shrimp"
  },
  {
    "dish_cuisine": "Convenience store",
    "dish_base": "Orajel",
    "dish_flavor": "Max Strength"
  },
  {
    "dish_cuisine": "American restaurant",
    "dish_base": "Juice",
    "dish_flavor": "Orange"
  },
  {
    "dish_cuisine": "Mediterranean restaurant",
    "dish_base": "Soup",
    "dish_flavor": "Lentil"
  },
  {
    "dish_cuisine": "Dessert restaurant",
    "dish_base": "Cupcake Box",
    "dish_flavor": "Six"
  },
  {
    "dish_cuisine": "American restaurant",
    "dish_base": "Cake",
    "dish_flavor": "Molten Chocolate"
  },
  {
    "dish_cuisine": "Bakery restaurant",
    "dish_base": "Croissant",
    "dish_flavor": "Chocolate"
  },
  {
    "dish

## Generate mapping table

In [39]:
# df_sample = df.sample(n=50, random_state=123).reset_index(drop=True)
# df_sample = df_sample[["restaurant_name", "restaurant_type", "menu_name"]]
# df_sample.head()

In [28]:
response

'[\n  {\n    "dish_cuisine": "Mexican restaurant",\n    "dish_base": "Quesadilla",\n    "dish_flavor": "Chicken"\n  },\n  {\n    "dish_cuisine": "American restaurant",\n    "dish_base": "Slider",\n    "dish_flavor": "Kid\'s"\n  },\n  {\n    "dish_cuisine": "Chinese restaurant",\n    "dish_base": "Lo Mein",\n    "dish_flavor": "Shrimp"\n  },\n  {\n    "dish_cuisine": "Convenience store",\n    "dish_base": "Orajel",\n    "dish_flavor": "Max Strength"\n  },\n  {\n    "dish_cuisine": "American restaurant",\n    "dish_base": "Juice",\n    "dish_flavor": "Orange"\n  },\n  {\n    "dish_cuisine": "Mediterranean restaurant",\n    "dish_base": "Soup",\n    "dish_flavor": "Lentil"\n  },\n  {\n    "dish_cuisine": "Dessert restaurant",\n    "dish_base": "Cupcake Box",\n    "dish_flavor": "Six"\n  },\n  {\n    "dish_cuisine": "American restaurant",\n    "dish_base": "Cake",\n    "dish_flavor": "Molten Chocolate"\n  },\n  {\n    "dish_cuisine": "Bakery restaurant",\n    "dish_base": "Croissant",\n   

In [38]:
import json
records = json.loads(response)
df_extracted = pd.DataFrame(records)
df_extracted

,dish_cuisine,dish_base,dish_flavor
0,Mexican restaurant,Quesadilla,Chicken
1,American restaurant,Slider,Kid's
2,Chinese restaurant,Lo Mein,Shrimp
3,Convenience store,Orajel,Max Strength
4,American restaurant,Juice,Orange
5,Mediterranean restaurant,Soup,Lentil
6,Dessert restaurant,Cupcake Box,Six
7,American restaurant,Cake,Molten Chocolate
8,Bakery restaurant,Croissant,Chocolate
9,American restaurant,Fry,Thai Fighter


In [42]:
dish_base_terms = df_extracted["dish_base"].dropna().str.lower().str.strip().unique().tolist()
dish_flavor_terms = df_extracted["dish_flavor"].dropna().str.lower().str.strip().unique().tolist()
dish_flavor_terms

['chicken',
 "kid's",
 'shrimp',
 'max strength',
 'orange',
 'lentil',
 'six',
 'molten chocolate',
 'chocolate',
 'thai fighter']

### Test Prompt
You are a normalization assistant. Given a list of food-related terms, group synonymous variants and map each to a single standardized name. Return a JSON object where each key is the original term (lowercased), and the value is the chosen standardized form (also lowercased). Use singular form, capitalize proper nouns (e.g. BBQ), and normalize common variants (e.g. 'hamburger' - 'burger', 'eggs' - 'egg').

In [44]:
def build_mapping_prompt(term_list: list[str], label: str) -> str:
    joined_terms = "\n".join(sorted(term_list))
    return (
        f"You are a normalization assistant.\n"
        f"The following is a list of raw {label} terms.\n"
        f"Map each to a short, clean, standardized {label} name.\n\n"
        "Rules:\n"
        "- Use singular form (e.g., 'eggs' → 'egg')\n"
        "- Use American English\n"
        "- Normalize abbreviations (e.g., 'bbq' → 'BBQ')\n"
        "- Capitalize proper nouns (e.g., 'McChicken')\n"
        "- Keep terms short and clean\n\n"
        f"{label.title()} terms:\n"
        "```text\n" + joined_terms + "\n```\n\n"
        "Return a JSON dictionary like:\n"
        "```json\n"
        "{\n  \"shrimp\": \"Shrimp\",\n  \"molten chocolate\": \"Chocolate\" \n}\n```"
    )

In [45]:
base_prompt = build_mapping_prompt(dish_base_terms, label="dish base")
flavor_prompt = build_mapping_prompt(dish_flavor_terms, label="dish flavor")
base_prompt

'You are a normalization assistant.\nThe following is a list of raw dish base terms.\nMap each to a short, clean, standardized dish base name.\n\nRules:\n- Use singular form (e.g., \'eggs\' → \'egg\')\n- Use American English\n- Normalize abbreviations (e.g., \'bbq\' → \'BBQ\')\n- Capitalize proper nouns (e.g., \'McChicken\')\n- Keep terms short and clean\n\nDish Base terms:\n```text\ncake\ncroissant\ncupcake box\nfry\njuice\nlo mein\norajel\nquesadilla\nslider\nsoup\n```\n\nReturn a JSON dictionary like:\n```json\n{\n  "shrimp": "Shrimp",\n  "molten chocolate": "Chocolate" \n}\n```'